In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast_models(df: pd.DataFrame) -> dict:
    """
    Compare TFT vs Snowflake forecast models.
    
    Required columns:
        DATES, PARENT_DEALER_CODE, SKU,
        PREDICTED_SALES_SKU_TFT, PREDICTED_SALES_SKU_SNOWFLAKE,
        ACTUAL_NET_SALES, SKU_TYPE_TFT_FORECAST
    """

    # ── helpers ──────────────────────────────────────────────────────────────
    def mape(actual, predicted, epsilon=1e-9):
        mask = actual.abs() > epsilon
        return (((actual[mask] - predicted[mask]).abs() / actual[mask].abs()) * 100).mean()

    def wmape(actual, predicted):
        """Weighted MAPE — avoids explosion on near-zero actuals."""
        total_actual = actual.abs().sum()
        if total_actual == 0:
            return np.nan
        return ((actual - predicted).abs().sum() / total_actual) * 100

    def bias_pct(actual, predicted):
        """Positive = over-forecast, Negative = under-forecast."""
        total_actual = actual.abs().sum()
        if total_actual == 0:
            return np.nan
        return ((predicted - actual).sum() / total_actual) * 100

    def compute_metrics(grp, actual_col, pred_col):
        actual    = grp[actual_col]
        predicted = grp[pred_col]
        n         = len(grp)
        mae       = mean_absolute_error(actual, predicted)
        rmse      = np.sqrt(mean_squared_error(actual, predicted))
        return {
            "N":           n,
            "MAE":         round(mae, 2),
            "RMSE":        round(rmse, 2),
            "MAPE":        round(mape(actual, predicted), 2),
            "wMAPE":       round(wmape(actual, predicted), 2),
            "Bias%":       round(bias_pct(actual, predicted), 2),
        }

    def compare_segment(grp, label):
        tft  = compute_metrics(grp, "ACTUAL_NET_SALES", "PREDICTED_SALES_SKU_TFT")
        snow = compute_metrics(grp, "ACTUAL_NET_SALES", "PREDICTED_SALES_SKU_SNOWFLAKE")
        return {"Segment": label, "Model": "TFT",       **tft}, \
               {"Segment": label, "Model": "Snowflake", **snow}

    results = []

    # ── 1. Overall ───────────────────────────────────────────────────────────
    r1, r2 = compare_segment(df, "Overall")
    results.extend([r1, r2])

    # ── 2. By SKU Type ───────────────────────────────────────────────────────
    for sku_type, grp in df.groupby("SKU_TYPE_TFT_FORECAST"):
        r1, r2 = compare_segment(grp, f"SKUType:{sku_type}")
        results.extend([r1, r2])

    # ── 3. By Month ──────────────────────────────────────────────────────────
    df["Month"] = pd.to_datetime(df["DATES"]).dt.to_period("M").astype(str)
    for month, grp in df.groupby("Month"):
        r1, r2 = compare_segment(grp, f"Month:{month}")
        results.extend([r1, r2])

    metrics_df = pd.DataFrame(results)

    # ── 4. Verdict logic ─────────────────────────────────────────────────────
    def verdict(seg_label, primary_metric="wMAPE"):
        seg = metrics_df[metrics_df["Segment"] == seg_label].set_index("Model")
        if seg.empty or len(seg) < 2:
            return "Insufficient data"
        tft_score  = seg.loc["TFT",       primary_metric]
        snow_score = seg.loc["Snowflake", primary_metric]
        delta_pct  = abs(tft_score - snow_score) / (max(tft_score, snow_score) + 1e-9) * 100
        if delta_pct < 5:          # < 5% difference → no material winner
            return "TOO CLOSE TO CALL (< 5% gap) — use other criteria"
        winner = "TFT" if tft_score < snow_score else "Snowflake"
        return f"{winner}  (wMAPE: TFT={tft_score:.1f}%  Snowflake={snow_score:.1f}%)"

    verdicts = {
        "Overall":  verdict("Overall"),
        "Runner":   verdict("SKUType:Runner"),   # ← Business priority
        "Repeater": verdict("SKUType:Repeater"),
        "Stranger": verdict("SKUType:Stranger"),
    }

    # ── 5. Final recommendation ──────────────────────────────────────────────
    runner_verdict   = verdicts["Runner"]
    overall_verdict  = verdicts["Overall"]

    if "Runner" in runner_verdict and "TOO CLOSE" not in runner_verdict:
        primary_winner = runner_verdict.split()[0]
    elif "Snowflake" in overall_verdict or "TFT" in overall_verdict:
        primary_winner = overall_verdict.split()[0]
    else:
        primary_winner = "INCONCLUSIVE"

    recommendation = {
        "RECOMMENDED_MODEL": primary_winner,
        "REASONING": (
            "Runner SKUs drive most revenue and are weighted highest. "
            f"Runner verdict: {runner_verdict}. "
            f"Overall verdict: {overall_verdict}."
        ),
        "CAVEAT": (
            "Bias% matters too — a low wMAPE with large Bias% means the model "
            "is systematically wrong in one direction. Check both."
        ),
    }

    return {
        "metrics_table":    metrics_df,
        "verdicts":         verdicts,
        "recommendation":   recommendation,
    }


# ── Run ───────────────────────────────────────────────────────────────────────
output = evaluate_forecast_models(df)

print("=" * 65)
print("METRICS BY SEGMENT")
print("=" * 65)
print(output["metrics_table"].to_string(index=False))

print("\n" + "=" * 65)
print("VERDICTS")
print("=" * 65)
for seg, v in output["verdicts"].items():
    flag = " ◄ PRIORITY" if seg == "Runner" else ""
    print(f"  {seg:<12}: {v}{flag}")

print("\n" + "=" * 65)
print("FINAL RECOMMENDATION")
print("=" * 65)
rec = output["recommendation"]
print(f"  Model      : {rec['RECOMMENDED_MODEL']}")
print(f"  Reasoning  : {rec['REASONING']}")
print(f"  ⚠ Caveat   : {rec['CAVEAT']}")